In [2]:
from pathlib import Path
PROC = Path("/content/week3_project/data/processed")
PROC.mkdir(parents=True, exist_ok=True)
print("Upload the two files into:", PROC)

Upload the two files into: /content/week3_project/data/processed


In [3]:
for f in sorted(PROC.iterdir()):
    print(f.name, f.stat().st_size, "bytes")

prepared_text_S01.jsonl 139417 bytes
prepared_text_S02.jsonl 282531 bytes


In [4]:
# Topic 4 — Data quality audit on the prepared chunks (S01 and S02)
import json, re, random, collections
from pathlib import Path
from datetime import date

PROC = Path("/content/week3_project/data/processed")
FILES = {"S01": PROC / "prepared_text_S01.jsonl", "S02": PROC / "prepared_text_S02.jsonl"}
META = {
    "S01": {"file": "IRBM-e-Invoice-Guideline.pdf", "pages": 62, "version": "4.8", "published": date(2026, 8, 30)},
    "S02": {"file": "irbm-e-invoice-specific-guideline.pdf", "pages": 132, "version": "4.7", "published": date(2026, 4, 20)},
}
REQUIRED = ["record_id", "text", "source", "source_sha256", "document_version", "published", "page"]
norm = lambda s: re.sub(r"\s+", " ", s).strip()

# ---------- Load ----------
data = {}
for sid, p in FILES.items():
    assert p.is_file(), f"Missing {p}. Upload it to data/processed or rerun the pipeline notebook."
    with p.open(encoding="utf-8") as fh:
        data[sid] = [json.loads(l) for l in fh if l.strip()]
    print(sid, "records loaded:", len(data[sid]))



S01 records loaded: 194
S02 records loaded: 382


In [11]:
# ---------- Sample of 16 (8 per source): first chunk, shortest, most table-like, 5 random ----------
def table_score(t):
    lines = [l for l in t.splitlines() if l.strip()]
    return sum(len(l.split()) <= 3 for l in lines) / max(len(lines), 1)

random.seed(42)
sample = []
for sid, recs in data.items():
    picks = {}
    picks[recs[0]["record_id"]] = "first chunk (front matter)"
    shortest = min(recs, key=lambda r: len(r["text"]))
    picks.setdefault(shortest["record_id"], "shortest chunk")
    tabley = max(recs, key=lambda r: table_score(r["text"]))
    picks.setdefault(tabley["record_id"], "most table-like chunk")
    rest = [r for r in recs if r["record_id"] not in picks]
    for r in random.sample(rest, 8 - len(picks)):
        picks[r["record_id"]] = "random"
    by_id = {r["record_id"]: r for r in recs}
    sample += [(by_id[i], k) for i, k in picks.items()]

print("\n" + "=" * 70 + "\nSAMPLE OF", len(sample), "CHUNKS\n" + "=" * 70)
for r, kind in sample:
    print(f"{r['record_id']} | page {r['page']} | {len(r['text'])} chars | {kind}")
    print("   ", repr(r["text"][:170]))




SAMPLE OF 16 CHUNKS
S01-0001 | page 1 | 182 chars | first chunk (front matter)
    'E-INVOICE GUIDELINE \nINLAND REVENUE BOARD OF MALAYSIA \nThis guideline is issued under section 134A of the Income Tax Act 1967\n\n(DATE OF PUBLICATION: 30 AUGUST 2026)\n\nTABL'
S01-0145 | page 45 | 6 chars | shortest chunk
    'y IRBM'
S01-0011 | page 2 | 51 chars | most table-like chunk
    '................................................ 58'
S01-0167 | page 53 | 500 chars | random
    'APPENDIX 2 – LIST OF MANDATORY AND OPTIONAL \nFIELDS UNDER ANNEXURE TO THE E-INVOICE \nAdditional mandatory fields to be included in Annexure to the e-Invoice \nApplicable t'
S01-0031 | page 9 | 163 chars | random
    'ancial reporting and processes to be \ndigitalised with industry standards, ensuring that MSMEs to adapt over a \nlonger period and mitigating potential disruptions.'
S01-0008 | page 2 | 500 chars | random
    '............................................................................ 31 \n2.4.3 Step

In [6]:
# ---------- Rule 1: provenance completeness ----------
print("\n" + "=" * 70 + "\nRULE 1 — every chunk has text, source, SHA-256, version, valid page\n" + "=" * 70)
rule1 = {}
for sid, recs in data.items():
    bad = []
    for r in recs:
        problems = [f"missing {k}" for k in REQUIRED if r.get(k) in (None, "")]
        if not isinstance(r.get("page"), int) or not (1 <= r["page"] <= META[sid]["pages"]):
            problems.append("page out of range")
        if r.get("document_version") != META[sid]["version"]:
            problems.append("version differs from expected")
        if problems:
            bad.append((r["record_id"], problems))
    rule1[sid] = bad
    print(sid, "chunks checked:", len(recs), "| failures:", len(bad),
          "| pass rate: %.1f%%" % (100 * (len(recs) - len(bad)) / len(recs)))
    for b in bad[:5]:
        print("   ", b)

# ---------- Rule 2: low-value chunks (table of contents, dotted lines, page numbers) ----------
print("\n" + "=" * 70 + "\nRULE 2 — flag low-value chunks (fewer than 40 letters or over 35% dots/digits/dashes)\n" + "=" * 70)
def noise_ratio(t):
    chars = [c for c in t if not c.isspace()]
    return sum(c in ".·_-" or c.isdigit() for c in chars) / max(len(chars), 1)
def low_value(t):
    return sum(c.isalpha() for c in t) < 40 or noise_ratio(t) > 0.35

rule2 = {}
for sid, recs in data.items():
    flagged = [r for r in recs if low_value(r["text"])]
    rule2[sid] = flagged
    pages = sorted({r["page"] for r in flagged})
    print(sid, "flagged:", len(flagged), "of", len(recs), "(%.1f%%)" % (100 * len(flagged) / len(recs)))
    print("   pages with flagged chunks:", pages[:30], "..." if len(pages) > 30 else "")
    for r in flagged[:3]:
        print("   e.g.", r["record_id"], "p.", r["page"], repr(r["text"][:90]))




RULE 1 — every chunk has text, source, SHA-256, version, valid page
S01 chunks checked: 194 | failures: 0 | pass rate: 100.0%
S02 chunks checked: 382 | failures: 0 | pass rate: 100.0%

RULE 2 — flag low-value chunks (fewer than 40 letters or over 35% dots/digits/dashes)
S01 flagged: 24 of 194 (12.4%)
   pages with flagged chunks: [1, 2, 5, 6, 8, 37, 45, 47, 54, 61] 
   e.g. S01-0002 p. 1 'TABLE OF CONTENT .........................................................................'
   e.g. S01-0003 p. 1 'FIGURES ..................................................................................'
   e.g. S01-0004 p. 1 'ting e-Invoice.......................................................................... 9'
S02 flagged: 39 of 382 (10.2%)
   pages with flagged chunks: [1, 2, 3, 4, 5, 7, 9, 12, 15, 28, 49, 51, 58, 59, 60, 68, 74, 89, 100, 107, 115, 117] 
   e.g. S02-0002 p. 1 'TABLE OF CONTENTS \nLIST OF FIGURES .......................................................'
   e.g. S02-0003 p. 1

In [9]:
# ---------- Five dimensions ----------
!pip install pypdf --quiet

print("\n" + "=" * 70 + "\nFIVE DIMENSIONS\n" + "=" * 70)

for sid, recs in data.items():   # Completeness
    pages_cov = {r["page"] for r in recs}
    missing_pages = sorted(set(range(1, META[sid]["pages"] + 1)) - pages_cov)
    print(f"[Completeness] {sid}: pages with at least one chunk: {len(pages_cov)}/{META[sid]['pages']} | pages with no chunk: {missing_pages[:10]}")

for sid, recs in data.items():   # Consistency
    ids = [r["record_id"] for r in recs]
    ident = {(r["source"], r["source_sha256"], r["document_version"], r["published"]) for r in recs}
    pages_seq = [r["page"] for r in recs]
    print(f"[Consistency] {sid}: unique record_ids {len(set(ids))}/{len(ids)} | distinct source/version/sha combos: {len(ident)} | pages in order: {pages_seq == sorted(pages_seq)}")
    variants = collections.Counter(m.group(0) for r in recs for m in re.finditer(r"[eE]\s?-\s?[iI][nN][vV][oO][iI][cC][eE]", r["text"]))
    print("   spelling variants of 'e-Invoice':", dict(variants.most_common(6)))

try:                             # Accuracy: sample chunks vs their source page
    from pypdf import PdfReader
    HEADER = re.compile(r"E-INVOICE (SPECIFIC )?GUIDELINE \(VERSION [\d.]+\)", re.I)
    def clean_page(t):
        t = HEADER.sub("", t); t = re.sub(r"[ \t]+", " ", t)
        return re.sub(r"^\s*\d+\s*$", "", t, flags=re.M).strip()
    pdf_text = {}
    for sid in data:
        pdf_path = RAW_DIR / META[sid]["file"]
        if pdf_path.is_file():
            pdf_text[sid] = {i + 1: norm(clean_page(p.extract_text() or "")) for i, p in enumerate(PdfReader(str(pdf_path)).pages)}
    if pdf_text:
        ok = sum(norm(r["text"]) in pdf_text[r["record_id"][:3]].get(r["page"], "") for r, _ in sample if r["record_id"][:3] in pdf_text)
        n = sum(r["record_id"][:3] in pdf_text for r, _ in sample)
        print(f"[Accuracy] sample chunks found verbatim in their source page after cleaning: {ok}/{n}")
    else:
        print("[Accuracy] raw PDFs not in Colab; skip the verbatim check (compare 3 sample chunks to the PDF by eye).")
except NameError:
    print("[Accuracy] RAW_DIR not defined; run Cell 1 first, or compare 3 sample chunks to the PDF by eye.")
for sid, recs in data.items():
    stray = [r["record_id"] for r in recs if re.search(r"\b[A-Za-z] -[A-Za-z]", r["text"])]
    print(f"[Accuracy] {sid}: chunks with a stray space before a hyphen (e.g. 'e -Invoice'): {len(stray)}", stray[:3])

today = date.today()             # Timeliness
for sid in data:
    age = (today - META[sid]["published"]).days
    print(f"[Timeliness] {sid}: version {META[sid]['version']}, published {META[sid]['published']}, {age} days old on {today}")
print("   Whether a newer version exists must be checked manually on hasil.gov.my.")

allrecs = [(sid, r) for sid, recs in data.items() for r in recs]   # Uniqueness
exact, loose = collections.defaultdict(list), collections.defaultdict(list)
for sid, r in allrecs:
    exact[norm(r["text"]).lower()].append((r["record_id"], r["page"]))
    loose[re.sub(r"[^a-z0-9]", "", r["text"].lower())].append((r["record_id"], r["page"]))
dup_exact = {k: v for k, v in exact.items() if len(v) > 1}
dup_loose = {k: v for k, v in loose.items() if len(v) > 1}
print(f"[Uniqueness] exact duplicate texts: {len(dup_exact)} groups, {sum(len(v) for v in dup_exact.values())} chunks")
print(f"[Uniqueness] duplicates ignoring case/punctuation: {len(dup_loose)} groups, {sum(len(v) for v in dup_loose.values())} chunks")
for k, v in list(dup_exact.items())[:3]:
    print("   e.g.", v[:4], repr(k[:70]))




FIVE DIMENSIONS
[Completeness] S01: pages with at least one chunk: 62/62 | pages with no chunk: []
[Completeness] S02: pages with at least one chunk: 132/132 | pages with no chunk: []
[Consistency] S01: unique record_ids 194/194 | distinct source/version/sha combos: 1 | pages in order: True
   spelling variants of 'e-Invoice': {'e-Invoice': 255, 'e -Invoice': 77, 'E-INVOICE': 9, 'e-invoice': 1}
[Consistency] S02: unique record_ids 382/382 | distinct source/version/sha combos: 1 | pages in order: True
   spelling variants of 'e-Invoice': {'e-Invoice': 599, 'e -Invoice': 118, 'E-INVOICE': 8, 'e -\nInvoice': 2, 'e-\ninvoice': 1}
[Accuracy] RAW_DIR not defined; run Cell 1 first, or compare 3 sample chunks to the PDF by eye.
[Accuracy] S01: chunks with a stray space before a hyphen (e.g. 'e -Invoice'): 56 ['S01-0012', 'S01-0021', 'S01-0022']
[Accuracy] S02: chunks with a stray space before a hyphen (e.g. 'e -Invoice'): 89 ['S02-0009', 'S02-0011', 'S02-0015']
[Timeliness] S01: version 4.8, 

In [10]:
# ---------- Coverage ----------
print("\n" + "=" * 70 + "\nCOVERAGE (keyword proxy: rough, not a proof of topic coverage)\n" + "=" * 70)
CATS = {
    "applicability / timeline": ["phase", "threshold", "exempt", "implementation", "timeline", "turnover"],
    "fields / document types": ["credit note", "debit note", "refund note", "self-billed", "consolidated", "mandatory"],
    "submission / validation / cancellation": ["validation", "validate", "submit", "submission", "cancel", "reject", "api"],
}
for cat, kws in CATS.items():
    for sid, recs in data.items():
        hit = [r for r in recs if any(k in r["text"].lower() for k in kws)]
        print(f"{cat:42s} {sid}: {len(hit):4d} chunks ({100 * len(hit) / len(recs):4.1f}%), {len({r['page'] for r in hit})} pages")

# ---------- Responsible data: personal-data pattern scan ----------
print("\n" + "=" * 70 + "\nPERSONAL-DATA PATTERN SCAN (counts only)\n" + "=" * 70)
PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "Malaysian mobile number": r"(?:\+?60|0)1\d[-\s]?\d{3,4}[-\s]?\d{4}",
    "MyKad-style number": r"\b\d{6}-\d{2}-\d{4}\b",
}
for name, pat in PATTERNS.items():
    for sid, recs in data.items():
        hits = [r for r in recs if re.search(pat, r["text"])]
        print(f"{name:26s} {sid}: {len(hits)} chunks", sorted({r['page'] for r in hits})[:8])

print("\n" + "=" * 70 + "\nSUMMARY\n" + "=" * 70)
for sid in data:
    print(f"{sid}: chunks={len(data[sid])} | rule1 failures={len(rule1[sid])} | rule2 flagged={len(rule2[sid])}")


COVERAGE (keyword proxy: rough, not a proof of topic coverage)
applicability / timeline                   S01:   27 chunks (13.9%), 14 pages
applicability / timeline                   S02:   19 chunks ( 5.0%), 16 pages
fields / document types                    S01:   38 chunks (19.6%), 25 pages
fields / document types                    S02:  136 chunks (35.6%), 72 pages
submission / validation / cancellation     S01:   75 chunks (38.7%), 36 pages
submission / validation / cancellation     S02:   64 chunks (16.8%), 46 pages

PERSONAL-DATA PATTERN SCAN (counts only)
email address              S01: 0 chunks []
email address              S02: 0 chunks []
Malaysian mobile number    S01: 0 chunks []
Malaysian mobile number    S02: 0 chunks []
MyKad-style number         S01: 0 chunks []
MyKad-style number         S02: 0 chunks []

SUMMARY
S01: chunks=194 | rule1 failures=0 | rule2 flagged=24
S02: chunks=382 | rule1 failures=0 | rule2 flagged=39
